# Comparaison de modèles

Superpose les contours Kp-vsys de plusieurs modèles moléculaires
(ou de plusieurs configurations d'analyse) sur un même panneau.

**Prérequis** : avoir exécuté `kpvsys_products.ipynb` pour chaque modèle
afin que les posteriors soient sauvegardés en NPZ dans `path_posteriors`.

**Workflow**
1. Lister les fichiers de posteriors et leurs étiquettes
2. Superposition des contours
3. Marginales Kp et vsys côte à côte

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from starships.plotting_fcts import plot_contours_overlay

## 1. Modèles à comparer

In [ ]:
path_posteriors = Path.home() / 'scratch/DataAnalysis/SPIRou/logl_grids/posteriors'

# Associer chaque fichier NPZ à une étiquette lisible
model_files = {
    'Modèle A': path_posteriors / 'modelA_posterior.npz',
    'Modèle B': path_posteriors / 'modelB_posterior.npz',
    # 'Modèle C': path_posteriors / 'modelC_posterior.npz',
}

# Position attendue du pic (lignes de référence)
kp_ref      = 227.15  # km/s
rv_expected = 0.0     # km/s

for label, path in model_files.items():
    status = '✓' if path.exists() else '✗ MANQUANT'
    print(f'  {status}  {label}: {path.name}')

## 2. Chargement des posteriors

In [ ]:
posteriors  = []
vsys_axes   = []
kp_axes     = []
margin_vsys_list = []
margin_kp_list   = []
labels = []

for label, path in model_files.items():
    data = np.load(path)
    posteriors.append(data['posterior'])
    vsys_axes.append(data['vsys_axis'])
    kp_axes.append(data['kp_axis'])
    margin_vsys_list.append(data['margin_vsys'])
    margin_kp_list.append(data['margin_kp'])
    labels.append(label)
    print(f'{label}: posterior {data["posterior"].shape}')

## 3. Contours superposés

Un jeu de contours par modèle, chacun avec sa couleur.
Un bon modèle place son contour centré sur la position attendue (lignes rouges).

In [ ]:
%matplotlib inline

fig, ax = plot_contours_overlay(
    posteriors, vsys_axes, kp_axes,
    labels=labels,
    n_sigma=1,
    # sigma_levels=[1., 2.],   # niveaux personnalisés
    # vsys_lim=(-30, 30),
    # kp_lim=(150, 300),
    rv_expected=rv_expected,
    kp_ref=kp_ref,
    figsize=(6, 5),
)
ax.set_title('Comparaison de modèles — contours 1σ', fontsize=12)
# fig.savefig('model_comparison_contours.pdf', bbox_inches='tight')

## 4. Marginales Kp et vsys côte à côte

In [ ]:
# Palette Paul Tol (même ordre que plot_contours_overlay)
_COLORS = ['#4477AA', '#EE6677', '#228833', '#CCBB44', '#66CCEE', '#AA3377']

fig, (ax_vsys, ax_kp) = plt.subplots(1, 2, figsize=(10, 4))

for i, (label, mv, mk, vsys, kp) in enumerate(zip(
    labels, margin_vsys_list, margin_kp_list, vsys_axes, kp_axes
)):
    color = _COLORS[i % len(_COLORS)]
    # Normaliser chaque marginale à son maximum pour faciliter la comparaison
    ax_vsys.plot(vsys, mv / mv.max(), color=color, lw=1.5, label=label)
    ax_kp.plot(mk / mk.max(), kp,    color=color, lw=1.5, label=label)

ax_vsys.axvline(rv_expected, color='gray', linestyle='--', lw=0.8)
ax_kp.axhline(kp_ref,        color='gray', linestyle='--', lw=0.8)

ax_vsys.set_xlabel(r'$v_\mathrm{sys}$ (km s$^{-1}$)', fontsize=13)
ax_vsys.set_ylabel('Posterior marginalisé (normalisé)', fontsize=11)
ax_vsys.legend(fontsize=10)

ax_kp.set_ylabel(r'$K_P$ (km s$^{-1}$)', fontsize=13)
ax_kp.set_xlabel('Posterior marginalisé (normalisé)', fontsize=11)

plt.tight_layout()
# fig.savefig('model_comparison_marginals.pdf', bbox_inches='tight')